# Offline RL Benchmark Notebook: BC, TD3+BC, CQL, AWAC, IQL, and GPR-IQL

This Colab notebook is a full experimental template for D4RL MuJoCo comparison experiments.

It trains and evaluates:

- Behavior Cloning (BC)
- TD3+BC
- Conservative Q-Learning (CQL)
- AWAC
- IQL
- GPR-IQL (5k GP supervision)
- GPR-IQL (10k GP supervision)
- Full GPR-IQL

The first five baselines use `d3rlpy`. The GPR-IQL section is implemented as a custom PyTorch IQL-style trainer with Gaussian Process target smoothing/pretraining.

> For quick Colab testing, keep `N_STEPS` small. For paper-grade results, use at least 300k--1M gradient steps and 3--5 seeds.


In [ ]:
# ============================================================
# 1. INSTALL DEPENDENCIES
# ============================================================

!pip install -q "gym==0.23.1" "d4rl @ git+https://github.com/Farama-Foundation/D4RL@master" "mujoco-py<2.2,>=2.1" "d3rlpy==1.1.1" scikit-learn pandas matplotlib tqdm


In [ ]:
# ============================================================
# 2. IMPORTS AND GLOBAL CONFIG
# ============================================================

import os
os.environ["D4RL_SUPPRESS_IMPORT_ERROR"] = "1"

import random
import time
from dataclasses import dataclass
from typing import Dict, List, Tuple

import gym
import d4rl
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel
from sklearn.preprocessing import StandardScaler

import d3rlpy
from d3rlpy.algos import BC, TD3PlusBC, CQL, AWAC, IQL
from d3rlpy.metrics.scorer import evaluate_on_environment

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
print("d3rlpy version:", d3rlpy.__version__)


In [ ]:
# ============================================================
# 3. EXPERIMENT SETTINGS
# ============================================================

ENVIRONMENTS = [
    "halfcheetah-medium-v2",
    "hopper-medium-v2",
    "walker2d-medium-v2",
    # Uncomment for full table:
    # "halfcheetah-medium-replay-v2",
    # "hopper-medium-replay-v2",
    # "walker2d-medium-replay-v2",
    # "halfcheetah-medium-expert-v2",
    # "hopper-medium-expert-v2",
    # "walker2d-medium-expert-v2",
]

SEEDS = [0]          # Use [0, 1, 2, 3, 4] for paper-grade experiments
N_STEPS = 50_000     # Use 300_000 to 1_000_000 for serious experiments
EVAL_EPISODES = 5

# GPR-IQL schedules
GPR_SCHEDULES = {
    "GPR-IQL (5k)": 5_000,
    "GPR-IQL (10k)": 10_000,
    "Full GPR-IQL": N_STEPS,
}

RESULT_DIR = "offline_rl_results"
os.makedirs(RESULT_DIR, exist_ok=True)


In [ ]:
# ============================================================
# 4. UTILITY FUNCTIONS
# ============================================================

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def d4rl_normalized_score(env, raw_score: float) -> float:
    """Returns D4RL normalized score in 0--100 style."""
    return env.get_normalized_score(raw_score) * 100.0

def evaluate_torch_policy(policy, env, episodes=5):
    policy.eval()
    raw_scores = []
    for _ in range(episodes):
        obs = env.reset()
        done = False
        total = 0.0
        while not done:
            obs_t = torch.tensor(obs, dtype=torch.float32, device=DEVICE).unsqueeze(0)
            with torch.no_grad():
                action = policy(obs_t).cpu().numpy()[0]
            obs, reward, done, _ = env.step(action)
            total += reward
        raw_scores.append(total)
    return float(np.mean(raw_scores))

def summarize_results(rows: List[Dict]) -> pd.DataFrame:
    df = pd.DataFrame(rows)
    summary = (
        df.groupby(["Environment", "Algorithm"])["NormalizedScore"]
        .agg(["mean", "std", "count"])
        .reset_index()
        .rename(columns={"mean": "Mean", "std": "Std", "count": "Seeds"})
    )
    return df, summary


In [ ]:
# ============================================================
# 5. BASELINE TRAINING USING d3rlpy
# ============================================================

def train_d3rlpy_baseline(env_name: str, algorithm_name: str, seed: int, n_steps: int):
    set_seed(seed)

    dataset, env = d3rlpy.datasets.get_d4rl(env_name)

    if algorithm_name == "BC":
        algo = BC(use_gpu=torch.cuda.is_available())
    elif algorithm_name == "TD3+BC":
        algo = TD3PlusBC(use_gpu=torch.cuda.is_available())
    elif algorithm_name == "CQL":
        algo = CQL(use_gpu=torch.cuda.is_available())
    elif algorithm_name == "AWAC":
        algo = AWAC(use_gpu=torch.cuda.is_available())
    elif algorithm_name == "IQL":
        algo = IQL(use_gpu=torch.cuda.is_available())
    else:
        raise ValueError(f"Unknown baseline: {algorithm_name}")

    algo.fit(
        dataset,
        n_steps=n_steps,
        n_steps_per_epoch=max(1, n_steps // 10),
        scorers={"environment": evaluate_on_environment(env, n_trials=EVAL_EPISODES)},
        experiment_name=f"{algorithm_name}_{env_name}_seed{seed}",
        save_interval=0,
        verbose=False,
    )

    raw_score = evaluate_on_environment(env, n_trials=EVAL_EPISODES)(algo)
    norm_score = d4rl_normalized_score(env, raw_score)

    return {
        "Environment": env_name,
        "Algorithm": algorithm_name,
        "Seed": seed,
        "RawScore": raw_score,
        "NormalizedScore": norm_score,
    }


In [ ]:
# ============================================================
# 6. CUSTOM GPR-IQL IMPLEMENTATION
# ============================================================

class MLP(nn.Module):
    def __init__(self, in_dim, out_dim, hidden=256, final_tanh=False):
        super().__init__()
        layers = [
            nn.Linear(in_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, out_dim),
        ]
        if final_tanh:
            layers.append(nn.Tanh())
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

class TwinQ(nn.Module):
    def __init__(self, state_dim, action_dim, hidden=256):
        super().__init__()
        self.q1 = MLP(state_dim + action_dim, 1, hidden)
        self.q2 = MLP(state_dim + action_dim, 1, hidden)

    def forward(self, s, a):
        x = torch.cat([s, a], dim=-1)
        return self.q1(x), self.q2(x)

    def min_q(self, s, a):
        q1, q2 = self.forward(s, a)
        return torch.minimum(q1, q2)

class DeterministicPolicy(nn.Module):
    def __init__(self, state_dim, action_dim, hidden=256):
        super().__init__()
        self.net = MLP(state_dim, action_dim, hidden, final_tanh=True)

    def forward(self, s):
        return self.net(s)

def expectile_loss(diff, tau=0.7):
    weight = torch.where(diff > 0, tau, 1 - tau)
    return weight * diff.pow(2)

def load_d4rl_tensors(env_name):
    env = gym.make(env_name)
    dataset = d4rl.qlearning_dataset(env)
    data = {
        "s": torch.tensor(dataset["observations"], dtype=torch.float32),
        "a": torch.tensor(dataset["actions"], dtype=torch.float32),
        "r": torch.tensor(dataset["rewards"], dtype=torch.float32).unsqueeze(-1),
        "ns": torch.tensor(dataset["next_observations"], dtype=torch.float32),
        "d": torch.tensor(dataset["terminals"], dtype=torch.float32).unsqueeze(-1),
    }
    return env, data

def fit_gp_targets(data, subset_size=2000):
    """Fit a lightweight GP on one-step reward targets over (s,a).
    This is intentionally small because exact GP is O(N^3).
    """
    n = data["s"].shape[0]
    idx = np.random.choice(n, size=min(subset_size, n), replace=False)
    x = torch.cat([data["s"][idx], data["a"][idx]], dim=-1).numpy()
    y = data["r"][idx].numpy().ravel()

    scaler = StandardScaler()
    x_scaled = scaler.fit_transform(x)

    kernel = ConstantKernel(1.0) * RBF(length_scale=1.0) + WhiteKernel(noise_level=1e-3)
    gp = GaussianProcessRegressor(
        kernel=kernel,
        alpha=1e-4,
        normalize_y=True,
        n_restarts_optimizer=1,
        random_state=0,
    )
    gp.fit(x_scaled, y)
    return gp, scaler

def predict_gp(gp, scaler, s, a):
    x = torch.cat([s, a], dim=-1).detach().cpu().numpy()
    x_scaled = scaler.transform(x)
    mu = gp.predict(x_scaled)
    return torch.tensor(mu, dtype=torch.float32, device=DEVICE).unsqueeze(-1)

def train_gpr_iql(env_name: str, seed: int, n_steps: int, gp_steps: int):
    set_seed(seed)
    env, data_cpu = load_d4rl_tensors(env_name)

    state_dim = data_cpu["s"].shape[1]
    action_dim = data_cpu["a"].shape[1]

    data = {k: v.to(DEVICE) for k, v in data_cpu.items()}

    q = TwinQ(state_dim, action_dim).to(DEVICE)
    v = MLP(state_dim, 1).to(DEVICE)
    pi = DeterministicPolicy(state_dim, action_dim).to(DEVICE)

    q_opt = torch.optim.Adam(q.parameters(), lr=3e-4)
    v_opt = torch.optim.Adam(v.parameters(), lr=3e-4)
    pi_opt = torch.optim.Adam(pi.parameters(), lr=3e-4)

    gamma = 0.99
    tau = 0.7
    beta = 3.0
    batch_size = 256

    print(f"Fitting GP for {env_name}, seed={seed}...")
    gp, scaler = fit_gp_targets(data_cpu, subset_size=2000)

    n = data["s"].shape[0]
    losses = []

    for step in tqdm(range(n_steps), desc=f"GPR-IQL gp_steps={gp_steps}"):
        idx = torch.randint(0, n, (batch_size,), device=DEVICE)
        s, a, r, ns, d = data["s"][idx], data["a"][idx], data["r"][idx], data["ns"][idx], data["d"][idx]

        with torch.no_grad():
            td_target = r + gamma * (1.0 - d) * v(ns)
            if step < gp_steps:
                gp_target = predict_gp(gp, scaler, s, a)
                target = 0.5 * td_target + 0.5 * gp_target
            else:
                target = td_target

        q1, q2 = q(s, a)
        q_loss = F.mse_loss(q1, target) + F.mse_loss(q2, target)
        q_opt.zero_grad()
        q_loss.backward()
        q_opt.step()

        with torch.no_grad():
            q_min = q.min_q(s, a)
        v_pred = v(s)
        v_loss = expectile_loss(q_min - v_pred, tau).mean()
        v_opt.zero_grad()
        v_loss.backward()
        v_opt.step()

        with torch.no_grad():
            adv = q.min_q(s, a) - v(s)
            weights = torch.exp(beta * adv).clamp(max=100.0)
        pred_a = pi(s)
        pi_loss = (weights * (pred_a - a).pow(2).mean(dim=-1, keepdim=True)).mean()
        pi_opt.zero_grad()
        pi_loss.backward()
        pi_opt.step()

        if step % 1000 == 0:
            losses.append({
                "step": step,
                "q_loss": float(q_loss.detach().cpu()),
                "v_loss": float(v_loss.detach().cpu()),
                "pi_loss": float(pi_loss.detach().cpu()),
            })

    raw_score = evaluate_torch_policy(pi, env, episodes=EVAL_EPISODES)
    norm_score = d4rl_normalized_score(env, raw_score)

    return {
        "Environment": env_name,
        "Algorithm": f"GPR-IQL ({'Full' if gp_steps == n_steps else str(gp_steps//1000)+'k'})",
        "Seed": seed,
        "RawScore": raw_score,
        "NormalizedScore": norm_score,
    }, pd.DataFrame(losses)


In [ ]:
# ============================================================
# 7. RUN BASELINE EXPERIMENTS
# ============================================================

BASELINES = ["BC", "TD3+BC", "CQL", "AWAC", "IQL"]

all_rows = []

for env_name in ENVIRONMENTS:
    for seed in SEEDS:
        for algo_name in BASELINES:
            print(f"
Running {algo_name} | {env_name} | seed={seed}")
            try:
                row = train_d3rlpy_baseline(env_name, algo_name, seed, N_STEPS)
                all_rows.append(row)
                pd.DataFrame(all_rows).to_csv(f"{RESULT_DIR}/raw_results_partial.csv", index=False)
                print(row)
            except Exception as e:
                print(f"FAILED: {algo_name} on {env_name}, seed={seed}: {e}")


In [ ]:
# ============================================================
# 8. RUN GPR-IQL EXPERIMENTS
# ============================================================

for env_name in ENVIRONMENTS:
    for seed in SEEDS:
        for label, gp_steps in GPR_SCHEDULES.items():
            print(f"
Running {label} | {env_name} | seed={seed}")
            try:
                row, loss_df = train_gpr_iql(env_name, seed, N_STEPS, gp_steps)
                row["Algorithm"] = label
                all_rows.append(row)
                loss_df.to_csv(f"{RESULT_DIR}/losses_{label.replace(' ', '_').replace('/', '_')}_{env_name}_seed{seed}.csv", index=False)
                pd.DataFrame(all_rows).to_csv(f"{RESULT_DIR}/raw_results_partial.csv", index=False)
                print(row)
            except Exception as e:
                print(f"FAILED: {label} on {env_name}, seed={seed}: {e}")


In [ ]:
# ============================================================
# 9. FINAL RESULT TABLE
# ============================================================

raw_df, summary_df = summarize_results(all_rows)

raw_path = f"{RESULT_DIR}/raw_results.csv"
summary_path = f"{RESULT_DIR}/summary_results.csv"

raw_df.to_csv(raw_path, index=False)
summary_df.to_csv(summary_path, index=False)

print("Raw results saved to:", raw_path)
print("Summary saved to:", summary_path)

display(raw_df)
display(summary_df)


In [ ]:
# ============================================================
# 10. PIVOT TABLE LIKE PAPER TABLE
# ============================================================

order = [
    "BC", "TD3+BC", "CQL", "AWAC", "IQL",
    "GPR-IQL (5k)", "GPR-IQL (10k)", "Full GPR-IQL"
]

pivot = summary_df.pivot(index="Environment", columns="Algorithm", values="Mean")
existing_order = [c for c in order if c in pivot.columns]
pivot = pivot[existing_order]
pivot.loc["Average"] = pivot.mean(axis=0)

display(pivot.round(2))
pivot.to_csv(f"{RESULT_DIR}/paper_style_table.csv")


In [ ]:
# ============================================================
# 11. BAR PLOT OF AVERAGE NORMALIZED SCORES
# ============================================================

avg = pivot.loc["Average"].dropna()

plt.figure(figsize=(12, 5))
plt.bar(avg.index, avg.values)
plt.ylabel("Average D4RL Normalized Score")
plt.xlabel("Algorithm")
plt.title("Average Offline RL Performance Across Environments")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.savefig(f"{RESULT_DIR}/average_comparison.png", dpi=300)
plt.show()


## Notes for paper-grade experiments

For a final paper table, change the settings near the top:

```python
SEEDS = [0, 1, 2, 3, 4]
N_STEPS = 500_000
ENVIRONMENTS = [
    "halfcheetah-medium-v2",
    "hopper-medium-v2",
    "walker2d-medium-v2",
    "halfcheetah-medium-replay-v2",
    "hopper-medium-replay-v2",
    "walker2d-medium-replay-v2",
    "halfcheetah-medium-expert-v2",
    "hopper-medium-expert-v2",
    "walker2d-medium-expert-v2",
]
```

The custom GPR-IQL block is intentionally transparent so that you can modify the GP target, the mixture coefficient, the kernel, and the supervision schedule.
